# ✈️ Airline Delay Analysis using SQL

## Project Objective

This notebook demonstrates SQL-based exploratory data analysis on a large-scale airline operations dataset using DuckDB.

### Tools Used
- SQL (DuckDB)
- Python
- Pandas

### Dataset
- 1.9 Million+ Flight Records
- 26 Features

Author: Harshita Chhabra

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import csv

df = pd.read_csv(
  '/content/drive/MyDrive/flight_data_2024.csv',
  sep=',',
  quoting=csv.QUOTE_NONE,
  on_bad_lines='skip'
)

# Clean column names by replacing all double quotes
df.columns = [col.replace('"', '') for col in df.columns]

df.head()

,month,day_of_week,dep_del15,dep_time_blk,distance_group,segment_number,concurrent_flights,number_of_seats,carrier_name,airport_flights_month,...,plane_age,departing_airport,latitude,longitude,previous_airport,prcp,snow,snwd,tmax,awnd
0,"""1",7,0,0800-0859,2.0,1.0,25.0,143.0,Southwest Airlines Co.,13056.0,...,8.0,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""
1,"""1",7,0,0700-0759,7.0,1.0,29.0,191.0,Delta Air Lines Inc.,13056.0,...,3.0,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""
2,"""1",7,0,0600-0659,7.0,1.0,27.0,199.0,Delta Air Lines Inc.,13056.0,...,18.0,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""
3,"""1",7,0,0600-0659,9.0,1.0,27.0,180.0,Delta Air Lines Inc.,13056.0,...,2.0,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""
4,"""1",7,0,0001-0559,7.0,1.0,10.0,182.0,Spirit Air Lines,13056.0,...,1.0,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""


In [3]:
!pip install duckdb

In [4]:
import duckdb
import pandas as pd

In [5]:
con = duckdb.connect()
con.register("flights", df)

In [6]:
con.sql("PRAGMA table_info('flights');").df()

,cid,name,type,notnull,dflt_value,pk
0,0,month,VARCHAR,False,None,False
1,1,day_of_week,BIGINT,False,None,False
2,2,dep_del15,BIGINT,False,None,False
3,3,dep_time_blk,VARCHAR,False,None,False
4,4,distance_group,DOUBLE,False,None,False
5,5,segment_number,DOUBLE,False,None,False
6,6,concurrent_flights,DOUBLE,False,None,False
7,7,number_of_seats,DOUBLE,False,None,False
8,8,carrier_name,VARCHAR,False,None,False
9,9,airport_flights_month,DOUBLE,False,None,False


### Q1. Total Number of Flights

In [7]:
con.sql("""
SELECT COUNT(*) AS total_flights
FROM flights;
""").df()

,total_flights
0,997547


### Q2. Total Airlines

con.sql("""
SELECT COUNT(DISTINCT carrier_name) AS total_airlines
FROM flights;
""").df()

In [8]:
con.sql("""
SELECT COUNT(DISTINCT carrier_name) AS total_airlines
FROM flights;
""").df()

,total_airlines
0,16


### Q3. Total Airports

In [9]:
con.sql("""
SELECT COUNT(DISTINCT departing_airport) AS total_airports
FROM flights;
""").df()

,total_airports
0,86


# Section 2 : Flight Operations Analysis

Flights by Month

In [10]:
con.sql('SELECT month, COUNT(*) AS total_flights FROM flights GROUP BY month ORDER BY month;').df()

,month,total_flights
0,"""1",477222
1,"""2",428822
2,"""3",91503


Flights by Airline

In [11]:
con.sql("""
SELECT
    carrier_name,
    COUNT(*) AS total_flights
FROM flights
GROUP BY carrier_name
ORDER BY total_flights DESC;
""").df()

,carrier_name,total_flights
0,Southwest Airlines Co.,208050
1,American Airlines Inc.,161466
2,Delta Air Lines Inc.,142443
3,United Air Lines Inc.,93095
4,SkyWest Airlines Inc.,90226
5,JetBlue Airways,47817
6,Comair Inc.,40913
7,Alaska Airlines Inc.,37966
8,American Eagle Airlines Inc.,34945
9,Endeavor Air Inc.,33141


Flights by Departure Time

In [12]:
con.sql("""
SELECT
    dep_time_blk,
    COUNT(*) AS total_flights
FROM flights
GROUP BY dep_time_blk
ORDER BY dep_time_blk;
""").df()

,dep_time_blk,total_flights
0,0001-0559,20228
1,0600-0659,61488
2,0700-0759,64605
3,0800-0859,70463
4,0900-0959,61548
5,1000-1059,62287
6,1100-1159,62720
7,1200-1259,61501
8,1300-1359,55343
9,1400-1459,59947


# Section 3 : Delay Analysis

In [13]:
con.sql("""
SELECT
ROUND(AVG(dep_del15)*100,2) AS delay_percentage
FROM flights;
""").df()

,delay_percentage
0,19.4


Delay by Airline

In [14]:
con.sql("""
SELECT
    carrier_name,
    ROUND(AVG(dep_del15)*100,2) AS delay_percentage
FROM flights
GROUP BY carrier_name
ORDER BY delay_percentage DESC;
""").df()

,carrier_name,delay_percentage
0,JetBlue Airways,26.98
1,Frontier Airlines Inc.,24.48
2,Atlantic Southeast Airlines,22.71
3,American Eagle Airlines Inc.,21.97
4,SkyWest Airlines Inc.,21.81
5,Allegiant Air,21.66
6,Southwest Airlines Co.,21.37
7,Endeavor Air Inc.,19.59
8,United Air Lines Inc.,18.42
9,Mesa Airlines Inc.,18.21


Delay by Month

In [15]:
con.sql("""
SELECT
    month,
    ROUND(AVG(dep_del15)*100,2) AS delay_percentage
FROM flights
GROUP BY month
ORDER BY month;
""").df()

,month,delay_percentage
0,"""1",17.43
1,"""2",22.09
2,"""3",17.10


# Section 4 : Weather Analysis

In [16]:
con.sql("""
SELECT
CASE
WHEN prcp=0 THEN 'No Rain'
ELSE 'Rain'
END AS rainfall_status,
ROUND(AVG(dep_del15)*100,2) AS delay_percentage
FROM flights
GROUP BY rainfall_status;
""").df()

,rainfall_status,delay_percentage
0,No Rain,16.73
1,Rain,24.36


Snow vs Delay

In [17]:
con.sql("""
SELECT
CASE
WHEN snow=0 THEN 'No Snow'
ELSE 'Snow'
END AS snow_status,
ROUND(AVG(dep_del15)*100,2) AS delay_percentage
FROM flights
GROUP BY snow_status;
""").df()

,snow_status,delay_percentage
0,No Snow,18.37
1,Snow,31.91


# Section 5 : Advanced SQL

Rank Airlines

In [18]:
con.sql("""
SELECT
carrier_name,
ROUND(AVG(dep_del15)*100,2) AS delay_percentage,
RANK() OVER(
ORDER BY AVG(dep_del15) DESC
) AS delay_rank
FROM flights
GROUP BY carrier_name;
""").df()

,carrier_name,delay_percentage,delay_rank
0,JetBlue Airways,26.98,1
1,Frontier Airlines Inc.,24.48,2
2,Atlantic Southeast Airlines,22.71,3
3,American Eagle Airlines Inc.,21.97,4
4,SkyWest Airlines Inc.,21.81,5
5,Allegiant Air,21.66,6
6,Southwest Airlines Co.,21.37,7
7,Endeavor Air Inc.,19.59,8
8,United Air Lines Inc.,18.42,9
9,Mesa Airlines Inc.,18.21,10


Top Airports (CTE)

In [19]:
con.sql("""
WITH airport_traffic AS
(
SELECT
departing_airport,
COUNT(*) AS total_flights
FROM flights
GROUP BY departing_airport
)

SELECT *
FROM airport_traffic
ORDER BY total_flights DESC
LIMIT 10;
""").df()

,departing_airport,total_flights
0,Atlanta Municipal,60047
1,Douglas Municipal,50317
2,Chicago O'Hare International,46416
3,Dallas Fort Worth Regional,45109
4,Los Angeles International,36558
5,Stapleton International,35382
6,Phoenix Sky Harbor International,30767
7,Logan International,29859
8,LaGuardia,29543
9,San Francisco International,27032


# Conclusion

This notebook explored airline operational performance using SQL.

Key Insights:

- Flight traffic trends
- Delay distribution
- Weather impact
- Airport performance
- Airline comparison
- Ranking analysis

The SQL insights generated here will be used to build an interactive Power BI dashboard.

Airlines operating more than 50,000 flights

In [20]:
con.sql("""
SELECT
    carrier_name,
    COUNT(*) AS total_flights
FROM flights
GROUP BY carrier_name
HAVING COUNT(*) > 50000
ORDER BY total_flights DESC;
""").df()

,carrier_name,total_flights
0,Southwest Airlines Co.,208050
1,American Airlines Inc.,161466
2,Delta Air Lines Inc.,142443
3,United Air Lines Inc.,93095
4,SkyWest Airlines Inc.,90226


Airports with more than 20,000 flights

In [21]:
con.sql("""
SELECT
    departing_airport,
    COUNT(*) AS total_flights
FROM flights
GROUP BY departing_airport
HAVING COUNT(*) > 20000
ORDER BY total_flights DESC;
""").df()

,departing_airport,total_flights
0,Atlanta Municipal,60047
1,Douglas Municipal,50317
2,Chicago O'Hare International,46416
3,Dallas Fort Worth Regional,45109
4,Los Angeles International,36558
5,Stapleton International,35382
6,Phoenix Sky Harbor International,30767
7,Logan International,29859
8,LaGuardia,29543
9,San Francisco International,27032


Average plane age by airport

In [22]:
con.sql("""
SELECT
    departing_airport,
    ROUND(AVG(plane_age),2) AS avg_plane_age
FROM flights
GROUP BY departing_airport
ORDER BY avg_plane_age DESC;
""").df()

,departing_airport,avg_plane_age
0,Lihue Airport,15.22
1,Honolulu International,14.84
2,Atlanta Municipal,14.45
3,Kahului Airport,14.31
4,Norfolk International,14.27
...,...,...
81,Orlando International,10.11
82,San Diego International Lindbergh Fl,9.79
83,McCarran International,9.61
84,Portland International,9.56


Average seats by airline

In [23]:
con.sql("""
SELECT
    carrier_name,
    ROUND(AVG(number_of_seats),0) AS avg_seats
FROM flights
GROUP BY carrier_name
ORDER BY avg_seats DESC;
""").df()

,carrier_name,avg_seats
0,Frontier Airlines Inc.,191.0
1,Spirit Air Lines,183.0
2,Alaska Airlines Inc.,162.0
3,American Airlines Inc.,161.0
4,Delta Air Lines Inc.,160.0
5,Hawaiian Airlines Inc.,160.0
6,United Air Lines Inc.,158.0
7,Southwest Airlines Co.,152.0
8,JetBlue Airways,136.0
9,Allegiant Air,129.0


Peak concurrent flights by airport

In [24]:
con.sql("""
SELECT
    departing_airport,
    MAX(concurrent_flights) AS peak_concurrent_flights
FROM flights
GROUP BY departing_airport
ORDER BY peak_concurrent_flights DESC
LIMIT 10;
""").df()

,departing_airport,peak_concurrent_flights
0,Atlanta Municipal,94.0
1,Chicago O'Hare International,87.0
2,Dallas Fort Worth Regional,78.0
3,Detroit Metro Wayne County,69.0
4,Douglas Municipal,68.0
5,Stapleton International,67.0
6,Houston Intercontinental,64.0
7,Los Angeles International,62.0
8,Minneapolis-St Paul International,58.0
9,Phoenix Sky Harbor International,50.0


Aircraft age category

In [25]:
con.sql("""
SELECT
CASE
WHEN plane_age < 10 THEN 'New'
WHEN plane_age BETWEEN 10 AND 20 THEN 'Medium'
ELSE 'Old'
END AS aircraft_category,
COUNT(*) AS total_flights
FROM flights
GROUP BY aircraft_category;
""").df()

,aircraft_category,total_flights
0,New,378333
1,Old,78480
2,Medium,540734


Flight distance category

In [26]:
con.sql("""
SELECT
CASE
WHEN distance_group <= 3 THEN 'Short Haul'
WHEN distance_group <= 7 THEN 'Medium Haul'
ELSE 'Long Haul'
END AS flight_type,
COUNT(*) AS total_flights
FROM flights
GROUP BY flight_type;
""").df()

,flight_type,total_flights
0,Short Haul,538194
1,Medium Haul,366837
2,Long Haul,92516


Wind category vs delays

In [27]:
con.sql("""
SELECT
CASE
WHEN CAST(REPLACE(awnd, '"', '') AS DOUBLE) < 5 THEN 'Low Wind'
WHEN CAST(REPLACE(awnd, '"', '') AS DOUBLE) < 10 THEN 'Moderate Wind'
ELSE 'High Wind'
END AS wind_category,
ROUND(AVG(dep_del15)*100,2) AS delay_rate
FROM flights
GROUP BY wind_category;
""").df()

,wind_category,delay_rate
0,Low Wind,15.22
1,Moderate Wind,18.95
2,High Wind,22.28


Temperature category

In [28]:
con.sql("""
SELECT
CASE
WHEN tmax < 10 THEN 'Cold'
WHEN tmax < 25 THEN 'Moderate'
ELSE 'Hot'
END AS temperature_category,
COUNT(*) AS total_flights
FROM flights
GROUP BY temperature_category;
""").df()

,temperature_category,total_flights
0,Hot,959317
1,Cold,4098
2,Moderate,34132


Flight status

In [29]:
con.sql("""
SELECT
CASE
WHEN dep_del15 = 1 THEN 'Delayed'
ELSE 'On Time'
END AS flight_status,
COUNT(*) AS total_flights
FROM flights
GROUP BY flight_status;
""").df()

,flight_status,total_flights
0,Delayed,193543
1,On Time,804004


Flights with above-average plane age

In [30]:
con.sql("""
SELECT *
FROM flights
WHERE plane_age >
(
SELECT AVG(plane_age)
FROM flights
);
""").df()

,month,day_of_week,dep_del15,dep_time_blk,distance_group,segment_number,concurrent_flights,number_of_seats,carrier_name,airport_flights_month,...,plane_age,departing_airport,latitude,longitude,previous_airport,prcp,snow,snwd,tmax,awnd
0,"""1",7,0,0600-0659,7.0,1.0,27.0,199.0,Delta Air Lines Inc.,13056.0,...,18.0,McCarran International,36.080,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""
1,"""1",7,0,1200-1259,1.0,1.0,26.0,119.0,Alaska Airlines Inc.,13056.0,...,12.0,McCarran International,36.080,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""
2,"""1",7,0,0600-0659,8.0,1.0,27.0,187.0,American Airlines Inc.,13056.0,...,18.0,McCarran International,36.080,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""
3,"""1",7,0,0800-0859,3.0,1.0,25.0,142.0,United Air Lines Inc.,13056.0,...,22.0,McCarran International,36.080,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""
4,"""1",7,1,1000-1059,3.0,1.0,29.0,142.0,United Air Lines Inc.,13056.0,...,19.0,McCarran International,36.080,-115.152,NONE,0.0,0.0,0.0,65.0,"2.91"""
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516422,"""3",1,0,2100-2159,5.0,1.0,18.0,150.0,JetBlue Airways,9353.0,...,19.0,Fort Lauderdale-Hollywood International,26.074,-80.152,NONE,0.0,0.0,0.0,81.0,"8.28"""
516423,"""3",1,0,1800-1859,9.0,1.0,16.0,150.0,JetBlue Airways,9353.0,...,15.0,Fort Lauderdale-Hollywood International,26.074,-80.152,NONE,0.0,0.0,0.0,81.0,"8.28"""
516424,"""3",1,1,0800-0859,5.0,1.0,16.0,162.0,JetBlue Airways,9353.0,...,15.0,Fort Lauderdale-Hollywood International,26.074,-80.152,NONE,0.0,0.0,0.0,81.0,"8.28"""
516425,"""3",1,0,0800-0859,5.0,1.0,16.0,162.0,JetBlue Airways,9353.0,...,14.0,Fort Lauderdale-Hollywood International,26.074,-80.152,NONE,0.0,0.0,0.0,81.0,"8.28"""


Airlines above average flight count

In [31]:
con.sql("""
SELECT
carrier_name,
COUNT(*) AS total_flights
FROM flights
GROUP BY carrier_name
HAVING COUNT(*) >
(
SELECT AVG(cnt)
FROM
(
SELECT COUNT(*) AS cnt
FROM flights
GROUP BY carrier_name
)
);
""").df()

,carrier_name,total_flights
0,American Airlines Inc.,161466
1,SkyWest Airlines Inc.,90226
2,Delta Air Lines Inc.,142443
3,United Air Lines Inc.,93095
4,Southwest Airlines Co.,208050


Maximum concurrent flights

In [32]:
con.sql("""
SELECT *
FROM flights
WHERE concurrent_flights =
(
SELECT MAX(concurrent_flights)
FROM flights
);
""").df()

,month,day_of_week,dep_del15,dep_time_blk,distance_group,segment_number,concurrent_flights,number_of_seats,carrier_name,airport_flights_month,...,plane_age,departing_airport,latitude,longitude,previous_airport,prcp,snow,snwd,tmax,awnd
0,"""2",5,1,2100-2159,2.0,4.0,94.0,180.0,Delta Air Lines Inc.,28011.0,...,5.0,Atlanta Municipal,33.641,-84.427,Sacramento International,0.01,0.0,0.0,67.0,"15.43"""
1,"""2",5,0,2100-2159,3.0,4.0,94.0,157.0,Delta Air Lines Inc.,28011.0,...,20.0,Atlanta Municipal,33.641,-84.427,Dallas Fort Worth Regional,0.01,0.0,0.0,67.0,"15.43"""
2,"""2",5,0,2100-2159,3.0,4.0,94.0,180.0,Delta Air Lines Inc.,28011.0,...,4.0,Atlanta Municipal,33.641,-84.427,Jacksonville International,0.01,0.0,0.0,67.0,"15.43"""
3,"""2",5,1,2100-2159,3.0,4.0,94.0,76.0,Endeavor Air Inc.,28011.0,...,5.0,Atlanta Municipal,33.641,-84.427,Westchester County,0.01,0.0,0.0,67.0,"15.43"""
4,"""2",5,0,2100-2159,2.0,4.0,94.0,76.0,Endeavor Air Inc.,28011.0,...,5.0,Atlanta Municipal,33.641,-84.427,Gulfport-Biloxi International,0.01,0.0,0.0,67.0,"15.43"""
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,"""2",5,0,2100-2159,3.0,4.0,94.0,157.0,Delta Air Lines Inc.,28011.0,...,20.0,Atlanta Municipal,33.641,-84.427,Cincinnati/Northern Kentucky International,0.01,0.0,0.0,67.0,"15.43"""
88,"""2",5,0,2100-2159,4.0,4.0,94.0,149.0,Delta Air Lines Inc.,28011.0,...,27.0,Atlanta Municipal,33.641,-84.427,Richmond International,0.01,0.0,0.0,67.0,"15.43"""
89,"""2",5,0,2100-2159,3.0,4.0,94.0,180.0,Delta Air Lines Inc.,28011.0,...,1.0,Atlanta Municipal,33.641,-84.427,Kansas City International,0.01,0.0,0.0,67.0,"15.43"""
90,"""2",5,0,2100-2159,4.0,4.0,94.0,191.0,Delta Air Lines Inc.,28011.0,...,2.0,Atlanta Municipal,33.641,-84.427,McCarran International,0.01,0.0,0.0,67.0,"15.43"""


Airports with above-average seating capacity

In [33]:
con.sql("""
SELECT
departing_airport,
AVG(number_of_seats) AS avg_seats
FROM flights
GROUP BY departing_airport
HAVING AVG(number_of_seats) >
(
SELECT AVG(number_of_seats)
FROM flights
);
""").df()

,departing_airport,avg_seats
0,San Antonio International,144.240240
1,San Diego International Lindbergh Fl,150.205248
2,Louis Armstrong New Orleans International,149.384969
3,Lambert-St. Louis International,138.896694
4,Palm Beach International,156.712896
5,Tampa International,163.251910
6,San Francisco International,138.320139
7,Southwest Florida International,165.854219
8,William P Hobby,148.948666
9,Honolulu International,171.847031


Airlines with above-average delay rate

In [34]:
con.sql("""
SELECT
carrier_name,
AVG(dep_del15) AS delay_rate
FROM flights
GROUP BY carrier_name
HAVING AVG(dep_del15) >
(
SELECT AVG(dep_del15)
FROM flights
);
""").df()

,carrier_name,delay_rate
0,Allegiant Air,0.216593
1,American Eagle Airlines Inc.,0.219659
2,Endeavor Air Inc.,0.195920
3,Atlantic Southeast Airlines,0.227126
4,Southwest Airlines Co.,0.213747
5,Frontier Airlines Inc.,0.244842
6,JetBlue Airways,0.269820
7,SkyWest Airlines Inc.,0.218064


Row Number

In [35]:
con.sql("""
SELECT
carrier_name,
COUNT(*) AS total_flights,
ROW_NUMBER() OVER(ORDER BY COUNT(*) DESC) AS row_num
FROM flights
GROUP BY carrier_name;
""").df()

,carrier_name,total_flights,row_num
0,Southwest Airlines Co.,208050,1
1,American Airlines Inc.,161466,2
2,Delta Air Lines Inc.,142443,3
3,United Air Lines Inc.,93095,4
4,SkyWest Airlines Inc.,90226,5
5,JetBlue Airways,47817,6
6,Comair Inc.,40913,7
7,Alaska Airlines Inc.,37966,8
8,American Eagle Airlines Inc.,34945,9
9,Endeavor Air Inc.,33141,10


Rank Airlines

In [36]:
con.sql("""
SELECT
carrier_name,
COUNT(*) AS total_flights,
RANK() OVER(ORDER BY COUNT(*) DESC) AS ranking
FROM flights
GROUP BY carrier_name;
""").df()

,carrier_name,total_flights,ranking
0,Southwest Airlines Co.,208050,1
1,American Airlines Inc.,161466,2
2,Delta Air Lines Inc.,142443,3
3,United Air Lines Inc.,93095,4
4,SkyWest Airlines Inc.,90226,5
5,JetBlue Airways,47817,6
6,Comair Inc.,40913,7
7,Alaska Airlines Inc.,37966,8
8,American Eagle Airlines Inc.,34945,9
9,Endeavor Air Inc.,33141,10



Dense Rank

In [37]:
con.sql("""
SELECT
carrier_name,
COUNT(*) AS total_flights,
DENSE_RANK() OVER(ORDER BY COUNT(*) DESC) AS dense_rank
FROM flights
GROUP BY carrier_name;
""").df()

,carrier_name,total_flights,dense_rank
0,Southwest Airlines Co.,208050,1
1,American Airlines Inc.,161466,2
2,Delta Air Lines Inc.,142443,3
3,United Air Lines Inc.,93095,4
4,SkyWest Airlines Inc.,90226,5
5,JetBlue Airways,47817,6
6,Comair Inc.,40913,7
7,Alaska Airlines Inc.,37966,8
8,American Eagle Airlines Inc.,34945,9
9,Endeavor Air Inc.,33141,10


Running Total

In [38]:
con.sql("""
SELECT
month,
COUNT(*) AS flights,
SUM(COUNT(*)) OVER(ORDER BY month) AS running_total
FROM flights
GROUP BY month;
""").df()

,month,flights,running_total
0,"""1",477222,477222.0
1,"""2",428822,906044.0
2,"""3",91503,997547.0


Previous Month Flights

In [39]:
con.sql("""
SELECT
month,
COUNT(*) AS flights,
LAG(COUNT(*)) OVER(ORDER BY month) AS previous_month
FROM flights
GROUP BY month;
""").df()

,month,flights,previous_month
0,"""1",477222,<NA>
1,"""2",428822,477222
2,"""3",91503,428822


Next Month Flights

In [40]:
con.sql("""
SELECT
month,
COUNT(*) AS flights,
LEAD(COUNT(*)) OVER(ORDER BY month) AS next_month
FROM flights
GROUP BY month;
""").df()

,month,flights,next_month
0,"""1",477222,428822
1,"""2",428822,91503
2,"""3",91503,<NA>


Monthly Delay CTE

In [41]:

con.sql("""
WITH monthly_delay AS
(
SELECT
month,
AVG(dep_del15) AS delay_rate
FROM flights
GROUP BY month
)
SELECT *
FROM monthly_delay
ORDER BY delay_rate DESC;
""").df()

,month,delay_rate
0,"""2",0.220903
1,"""1",0.174275
2,"""3",0.171000


Airport Traffic CTE

In [42]:
con.sql("""
WITH airport_traffic AS
(
SELECT
departing_airport,
COUNT(*) AS flights
FROM flights
GROUP BY departing_airport
)
SELECT *
FROM airport_traffic
WHERE flights > 10000;
""").df()

,departing_airport,flights
0,Douglas Municipal,50317
1,San Diego International Lindbergh Fl,15167
2,Chicago O'Hare International,46416
3,Detroit Metro Wayne County,21823
4,John F. Kennedy International,20887
5,Tampa International,13092
6,Minneapolis-St Paul International,23114
7,San Francisco International,27032
8,Stapleton International,35382
9,Newark Liberty International,17659


Airline Seating CTE

In [43]:
con.sql("""
WITH airline_stats AS
(
SELECT
carrier_name,
AVG(number_of_seats) AS avg_seats
FROM flights
GROUP BY carrier_name
)
SELECT *
FROM airline_stats
ORDER BY avg_seats DESC;
""").df()

,carrier_name,avg_seats
0,Frontier Airlines Inc.,191.189690
1,Spirit Air Lines,182.842578
2,Alaska Airlines Inc.,161.572091
3,American Airlines Inc.,160.603687
4,Hawaiian Airlines Inc.,159.954126
5,Delta Air Lines Inc.,159.555871
6,United Air Lines Inc.,157.837521
7,Southwest Airlines Co.,151.846835
8,JetBlue Airways,136.083067
9,Allegiant Air,129.000000


Airline Delay CTE

In [44]:
con.sql("""
WITH delays AS
(
SELECT
carrier_name,
AVG(dep_del15) AS delay_rate
FROM flights
GROUP BY carrier_name
)
SELECT *
FROM delays
ORDER BY delay_rate DESC;
""").df()

,carrier_name,delay_rate
0,JetBlue Airways,0.269820
1,Frontier Airlines Inc.,0.244842
2,Atlantic Southeast Airlines,0.227126
3,American Eagle Airlines Inc.,0.219659
4,SkyWest Airlines Inc.,0.218064
5,Allegiant Air,0.216593
6,Southwest Airlines Co.,0.213747
7,Endeavor Air Inc.,0.195920
8,United Air Lines Inc.,0.184177
9,Mesa Airlines Inc.,0.182129


Average traffic by airline

In [45]:
con.sql("""
SELECT
carrier_name,
AVG(concurrent_flights) AS avg_traffic
FROM flights
GROUP BY carrier_name
ORDER BY avg_traffic DESC;
""").df()

,carrier_name,avg_traffic
0,American Eagle Airlines Inc.,34.665102
1,Delta Air Lines Inc.,33.876877
2,Comair Inc.,33.380490
3,Mesa Airlines Inc.,31.619790
4,American Airlines Inc.,30.908693
5,SkyWest Airlines Inc.,30.527996
6,Endeavor Air Inc.,30.350412
7,Atlantic Southeast Airlines,29.673616
8,United Air Lines Inc.,29.500392
9,Spirit Air Lines,23.693400


Average plane age by month

In [46]:
con.sql("""
SELECT
month,
AVG(plane_age) AS avg_plane_age
FROM flights
GROUP BY month;
""").df()

,month,avg_plane_age
0,"""1",11.759135
1,"""2",11.626768
2,"""3",11.529524


Maximum seating capacity

In [47]:
con.sql("""
SELECT
carrier_name,
MAX(number_of_seats) AS maximum_seats
FROM flights
GROUP BY carrier_name;
""").df()

,carrier_name,maximum_seats
0,Spirit Air Lines,228.0
1,Allegiant Air,129.0
2,Mesa Airlines Inc.,129.0
3,American Eagle Airlines Inc.,129.0
4,Endeavor Air Inc.,129.0
5,Atlantic Southeast Airlines,129.0
6,Southwest Airlines Co.,175.0
7,Delta Air Lines Inc.,306.0
8,Frontier Airlines Inc.,230.0
9,Alaska Airlines Inc.,185.0


Average rainfall by airport

In [48]:
con.sql("""
SELECT
departing_airport,
AVG(prcp) AS average_rainfall
FROM flights
GROUP BY departing_airport
ORDER BY average_rainfall DESC
LIMIT 10;
""").df()

,departing_airport,average_rainfall
0,McGhee Tyson,0.283193
1,Memphis International,0.224265
2,Sacramento International,0.207931
3,Birmingham Airport,0.207320
4,Metropolitan Oakland International,0.198413
5,Palm Beach International,0.184283
6,Standiford Field,0.181420
7,Long Beach Daugherty Field,0.176444
8,Atlanta Municipal,0.172530
9,Louis Armstrong New Orleans International,0.168391


Average wind speed by airlines

In [49]:
con.sql("""
SELECT
carrier_name,
AVG(CAST(REPLACE(awnd, '"', '') AS DOUBLE)) AS average_wind_speed
FROM flights
GROUP BY carrier_name
ORDER BY average_wind_speed DESC;
""").df()

,carrier_name,average_wind_speed
0,JetBlue Airways,10.078829
1,Endeavor Air Inc.,10.002808
2,American Eagle Airlines Inc.,9.898796
3,Atlantic Southeast Airlines,9.389428
4,United Air Lines Inc.,9.150450
5,Delta Air Lines Inc.,9.039888
6,SkyWest Airlines Inc.,8.995464
7,Spirit Air Lines,8.991597
8,American Airlines Inc.,8.762321
9,Frontier Airlines Inc.,8.568144


Average wind speed by airline